### Scikit-learn Tutorial 
This tutorial focuses on key elements of scikit-learn, including its design philosophy and usage principles. 
- Design principles of scikit-learn 
    - Consistency: All objects follow a uniform API: fit, predict, transform etc
    - Modularity: Algorithms are decoupled from data. Pipelines help to combine them 
    - Composition: Can chain transformers and models into workflows using Pipeline 
    - Non-proliferation of classes: Few, general purpose classes instead of many specialised ones 
    - Sensible Defaults: Many models work out-of-the-box with good defaults
- Core Objects/Interfacts (4 of them)
    - Estimator: anything with .fit() -> learns from the data 
        - E.g. LinearRegression, KMeans, PCA
    - Transformer: anything with .transform() -> modifies or projects data 
        - E.g. StandardScaler, TfidVectorizer 
    - Predictor: anything with .predict() -> makes predictions 
        - E.g. RandomForestClassifier, SVR
    - Meta-Estimators: Wrappers that modify other estimators 
        - E.g. GridSearchCV, RandomizedSearchCV, OnevsRestClassifier, BaggingClassifier, VotingClassifier, Pipeline
- Core Libraries and their functions 
    - sklearn.datasets -> Load built-in or external datasets 
    - sklearn.model_selection -> Cross-Validation, train-test split, hyperparameter tuning 
    - sklearn.preprocessing -> Feature scaling, encoding, normalisation 
    - sklearn.pipeline -> Chain multiple steps together 
    - sklearn.linear_model, sklearn.tree, sklearn.svm, sklearn.ensemble -> Machine Learning algorithms 
    - sklearn.metrics -> accuracy, precision, confusion matrix 
    - sklearn.decomposition -> dimensionality reduction (e.g. PCA)
    - sklearn.feature_selection -> select important features 
- Main workflow 
    - Raw Data (X, y) -> Preprocessing (e.g. StdScaler) -> Estimator (fit/predict/transform)
    - Wrap this all up in a pipeline that does 
        - X_temp = X_train 
        - for step in pipeline[:-1]: #all transformers 
            - X_temp = step.fit_transform(X_temp, y_train)
        - pipeline[-1].fit(X_temp, y_train )
- Under the hood, scikit-learn runs almost entirely on NumPy arrays 
    - Inputs like X and y are NumPy arrays, even if pass pandas Dataframes
    - Transformers and models operate on NumPy, not pandas 
    - Step-by-step breakdown of below code 
        - load_breast_cancer(return_X_y=True): returns X: numpy.ndarray, shape = (569,30). y: numpy.ndarray, shape=(569,)
        - train-test-split also returns numpy.ndarray
        - StandardScaler().fit_transform(X_train) is computed using Numpy broadcasting 
        - LogisticRegression().fit(): relies on both SciPy for optimisation and NumPy vector math 
    - Pandas not explicitly needed, but used more for EDA/Inspection/Feature selection can convert back and forth 
        - E.g. df = pd.DataFrame(X, columns=feature_names) / X = df.values
- Note that we use train_test_split first to split into train (for fitting of params and hyperparams tuning) vs testing set (held out only used for final eval). This is used to measure generalisation after model selection
    - If want to do hyperparameter tuning, use GridSearchCV which 
        - Splits X_train internally into k folds, trains on the k-1 fold and validates on the held-out fold, 
        - Rotates until all folds are tested to give a robust estimate of model performance corresponding to a combination of hyperparams
        - Then choose the best hyperparameters based on average CV score acros the folds
    - Then finally evaluate on the test set (true final evaluation), if not will risk overfitting
- Also note that the 3 core components of Hyperparameter Tuning in scikitlearn are 
    - Pipeline: Encapsulates the entire ML Workflow (scaling -> feature selection -> model) for a given hyperparameter combination
    - Param Grid: Dictionary that defines the hyperparameters and values to search 
    - GridSearchCV: object that actually performs the search, via cross-validation 
- What happens under the hood is that for each unique combo of hyperparams, scikit-learn (1) clones the pipeline, (2) applies one set of hyperparams, (3) runs CV, (4) stores the average CV score, then (5) at the end of all, chooses the best performing combo



In [3]:
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report, confusion_matrix

#load dataset and split into train vs test 
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.25, #betwee 0-1 (proportion). Can also specify absolute values
                                                    train_size=None, #proportion of absolute number of training samples. If only one is given, the other is computed automatically
                                                    random_state=42, #seed for reproducibility
                                                    shuffle=True, #Whether to shuffle before splitting
                                                    stratify = y) #Ensure class distribution is preserved in train/test sets (useful for classification)

#Create pipeline (List of Tuples:) -> one run for each combo of hyperparameters
pipe = Pipeline([ #pipeline first does StandardScaler().fit_transform(X_train), then does LogisticRegression().fit() on the scaled features
    ('scaler', StandardScaler()), #pipeline is a list of tuples: all until the last 1 is a transformer, and the last one if the estimator
    ('selector', SelectKBest(score_func=f_classif)), #Select top k features 
    ('clf', LogisticRegression()) #fit classifier 
])

#Define parameter grid for GridSearchCV 
param_grid = {
    'selector__k': [10, 15, 20, 'all'], 
    'clf__C': [0.01, 0.1, 1.0, 10.0], 
    'clf__penalty': ['l2'], 
    'clf__solver': ['lbfgs']

}

#Wrap pipeline with GridSearchCV
grid = GridSearchCV(pipe, param_grid=param_grid, cv=5, n_jobs=-1, scoring='accuracy')
grid.fit(X_train, y_train)

#Evaluate
print("Best Params:", grid.best_params_)
print("Best CV Score", grid.best_score_)

# Final evaluation on test set
y_pred = grid.predict(X_test)
print("\n📊 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\n📄 Classification Report:\n", classification_report(y_test, y_pred))

Best Params: {'clf__C': 0.1, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs', 'selector__k': 'all'}
Best CV Score 0.9765253077975377

📊 Confusion Matrix:
 [[51  2]
 [ 1 89]]

📄 Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.96      0.97        53
           1       0.98      0.99      0.98        90

    accuracy                           0.98       143
   macro avg       0.98      0.98      0.98       143
weighted avg       0.98      0.98      0.98       143



In [ ]:
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report, confusion_matrix

#load dataset and split into train vs test 
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.25, #betwee 0-1 (proportion). Can also specify absolute values
                                                    train_size=None, #proportion of absolute number of training samples. If only one is given, the other is computed automatically
                                                    random_state=42, #seed for reproducibility
                                                    shuffle=True, #Whether to shuffle before splitting
                                                    stratify = y) #Ensure class distribution is preserved in train/test sets (useful for classification)

#Create pipeline (List of Tuples:)
pipe = Pipeline([ #pipeline first does StandardScaler().fit_transform(X_train), then does LogisticRegression().fit() on the scaled features
    ('scaler', StandardScaler()), #pipeline is a list of tuples: all until the last 1 is a transformer, and the last one if the estimator
    ('clf', LogisticRegression()) #fit classifier 
])

#Fit and evaluate
pipe.fit(X_train, y_train) #based on MLE (i.e. minimize negative-log-likelihood function)
score = pipe.score(X_test, y_test)
y_pred = pipe.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

#Make inferences given some X_new; pipeline automatically scales it for you
X_new = X_test[0].reshape(1,-1)
pipe.predict(X_new)
pipe.predict_proba(X_new) #logistic regression uses sigmoid function; multi-class will use softmax 
pipe.predict_log_proba(X_new) #this just returns the log probabilities which are more numerically stable 

#Note: the above code is equivalent to the following steps (basic one with just scalar and classifier, excluding the grid search etc)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) #remember to scale both X_train and X_test, but no need to scale y (targets)
X_test_scaled = scaler.transform(X_test) #cannot fit again because we cannot refit the scalar on the test set, which causes data leakage. I.e. cannot give the model access to future information 
clf = LogisticRegression()
clf.fit(X_train_scaled, y_train)
score = clf.score(X_test_scaled, y_test)

## 📊 ML Model Evaluation Cheatsheet

---

### 🧪 1. Classification Metrics

| **Metric**        | **Formula**                                                                 | **Explanation**                                                   | **Use When**                              |
|-------------------|------------------------------------------------------------------------------|--------------------------------------------------------------------|-------------------------------------------|
| **Accuracy**       | \( \frac{TP + TN}{TP + TN + FP + FN} \)                                     | Proportion of total correct predictions.                           | Classes are balanced.                     |
| **Precision**      | \( \frac{TP}{TP + FP} \)                                                    | How many predicted positives were correct.                         | You care more about **false positives**.  |
| **Recall (Sensitivity)** | \( \frac{TP}{TP + FN} \)                                              | How many actual positives were caught.                            | You care more about **false negatives**.  |
| **F1 Score**       | \( \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} \) | Harmonic mean of precision and recall.                            | You want a **balance** of both.           |
| **ROC AUC**        | Area under the ROC curve                                                   | How well model ranks positives over negatives.                     | You want to compare classifiers.          |
| **Confusion Matrix**| Grid of TP / TN / FP / FN                                                 | Shows actual vs. predicted counts.                                | You want a **granular error view**.       |

---

### 📐 2. Regression Metrics

| **Metric**        | **Formula**                                                                 | **Explanation**                                                   | **Use When**                              |
|-------------------|------------------------------------------------------------------------------|--------------------------------------------------------------------|-------------------------------------------|
| **MAE**           | \( \frac{1}{n} \sum |y_i - \hat{y}_i| \)                                     | Average absolute error.                                            | Easy-to-interpret errors.                 |
| **MSE**           | \( \frac{1}{n} \sum (y_i - \hat{y}_i)^2 \)                                  | Penalizes larger errors more than MAE.                            | You want to **penalize big mistakes**.    |
| **RMSE**          | \( \sqrt{ \frac{1}{n} \sum (y_i - \hat{y}_i)^2 } \)                         | Same as MSE, but in same units as \( y \).                        | More interpretable than MSE.              |
| **R² Score**      | \( 1 - \frac{SS_{res}}{SS_{tot}} \), with:<br> \( SS_{res} = \sum (y_i - \hat{y}_i)^2 \) <br> \( SS_{tot} = \sum (y_i - \bar{y})^2 \) | Variance explained by model.        | You want to **measure goodness of fit**. |

---

### 🔁 3. Cross-Validation

| **Technique**     | **Explanation**                                                             | **Use Case**                               |
|-------------------|------------------------------------------------------------------------------|--------------------------------------------|
| **K-Fold**         | Split data into K parts. Rotate test set through folds.                     | General-purpose validation.                |
| **Stratified K-Fold**| Same as K-Fold, but preserves class ratios.                             | For **imbalanced classification**.         |
| **Leave-One-Out** | Use one sample as test, rest as train. Repeat for all.                      | Very small datasets.                        |
| **cross_val_score**| Runs a model across folds, returns metric scores.                         | Automate cross-validation scoring.         |

---

### 🔍 Explanation: What does `[:, 1]` mean?

In `model.predict_proba(X_test)[:, 1]`:

- `predict_proba(X_test)` returns a **2D array** of shape `(n_samples, n_classes)`.
- Each row gives the **probabilities** of the classes for one test sample.
- `[:, 1]` selects the **probability for class 1** (usually the *positive* class in binary classification).

📌 Example:
```python
[[0.8, 0.2],
 [0.3, 0.7]]

- model.predict_proba(X_test)[:,1] returns [0.2, 0.7]


In [ ]:
#Evaluate Model Function 

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,  f1_score, roc_auc_score, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.model_selection import cross_val_score
import numpy as np

def evaluate_model(model, X_train, y_train, X_test, y_test, task='classification'): 
    print(f"\n Evaluating {model.__class__.__name__} ({task})") #note: say we have model = RandomForestClassifier, model.__class__ returns <class 'sklearn.ensemble._forest.RandomForestClassifier'>
    #model.__class__.__name__ just returns RandomForestClassifier. I.e. the former gives the class type of the model instance, and the latter returns just the class name string

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if task == 'classification': 
        y_proba = model.predict_proba(X_test)[:,1] if hasattr(model, "predict_proba") else None #note that hasattr(obj, "attribute_name") checks if the object has a given attribute or method, and returns True or False
        print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}") #:.4f is a format specifier used in f-strings. ':' starts the format specifier, .4 rounds to 4 decimal places, and f is fixed-point notation (i.e. decimal format like 3.1416 instead of scientific notation 3.1416e+00)
        print(f"Precision: {precision_score(y_test, y_pred, average='binary'):.4f}")
        print(f"Recall:    {recall_score(y_test, y_pred, average='binary'):.4f}")
        print(f"F1 Score:  {f1_score(y_test, y_pred, average='binary'):.4f}")
        if y_proba is not None:
            print(f"ROC AUC:   {roc_auc_score(y_test, y_proba):.4f}")
        print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    elif task == 'regression':
        print(f"MAE:       {mean_absolute_error(y_test, y_pred):.4f}")
        print(f"MSE:       {mean_squared_error(y_test, y_pred):.4f}")
        print(f"RMSE:      {mean_squared_error(y_test, y_pred, squared=False):.4f}")
        print(f"R² Score:  {r2_score(y_test, y_pred):.4f}")

    else:
        raise ValueError("Task must be 'classification' or 'regression'")

    return model



## ROC Curve Cheatsheet

### 📌 What is ROC?
The **Receiver Operating Characteristic (ROC)** curve is a graph showing the **trade-off between sensitivity and specificity** for a binary classifier as the decision threshold is varied.

---

### 📐 Confusion Matrix Basics (for Binary Classification)

|                    | **Predicted: Positive** | **Predicted: Negative** |
|--------------------|-------------------------|--------------------------|
| **Actual: Positive** | True Positive (TP)       | False Negative (FN)       |
| **Actual: Negative** | False Positive (FP)      | True Negative (TN)        |

---

### 📊 ROC Curve Axes

- **X-axis (False Positive Rate)**:
  \[
  \text{FPR} = \frac{FP}{FP + TN}
  \]

- **Y-axis (True Positive Rate / Recall)**:
  \[
  \text{TPR} = \frac{TP}{TP + FN}
  \]

Each point on the ROC curve corresponds to a different **threshold** for deciding between class 0 or 1.

---

### 📏 AUC (Area Under Curve)

- **AUC = 1.0**: Perfect classifier
- **AUC = 0.5**: Random guessing (diagonal line)
- **AUC < 0.5**: Worse than random

---



In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# 1. Create synthetic data
X, y = make_classification(n_samples=1000, n_classes=2, random_state=42)

# 2. Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# 3. Train a classifier
model = LogisticRegression()
model.fit(X_train, y_train)

# 4. Predict probabilities (for class 1)
y_probs = model.predict_proba(X_test)[:, 1]

# 5. Calculate ROC metrics
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

# 6. Plot ROC curve
plt.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve')
plt.legend()
plt.grid()
plt.show()
